In [1]:
!git clone https://github.com/cg123/mergekit.git
%cd mergekit
%pip install -e .

Cloning into 'mergekit'...
remote: Enumerating objects: 3031, done.
remote: Counting objects: 100% (1230/1230), done.
remote: Compressing objects: 100% (361/361), done.
remote: Total 3031 (delta 1064), reused 869 (delta 869), pack-reused 1801 (from 3)
Receiving objects: 100% (3031/3031), 1.02 MiB | 14.99 MiB/s, done.
Resolving deltas: 100% (2081/2081), done.
/kaggle/working/mergekit
Obtaining file:///kaggle/working/mergekit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.

In [2]:
pip install --upgrade transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 23.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.29.0
    Uninstalling huggingface-hub-0.29.0:
      Successfully uninstalled huggingface-hub-0.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.0
    Uninstalling transformers-4.47.0:
      Successfully uninstalled transformers-4.47.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from huggingface_hub import login
login(token = "XXXXXXXXXXXXXXXXX") # replace with your own token

In [ ]:
yaml_config = """
merge_method: task_arithmetic
base_model: TheMockingJay1013/gemma-3-peft-safe
models:
  - model: google/gemma-3-1b-pt
    parameters:
      weight: 1.0 # alpha
  - model: TheMockingJay1013/gemma-3-sft-peft-dare
    parameters:
      weight: 1.0
"""

# Save config as yaml file
with open('config.yaml', 'w', encoding="utf-8") as f:
    f.write(yaml_config)

In [5]:
OUTPUT_PATH = "./merged"  # folder to store the result in
LORA_MERGE_CACHE = "/tmp"  # change if you want to keep these for some reason
CONFIG_YML = "config.yaml"  # merge configuration file
COPY_TOKENIZER = True  # you want a tokenizer? yeah, that's what i thought
LAZY_UNPICKLE = False  # experimental low-memory model loader
LOW_CPU_MEMORY = False  # enable if you somehow have more VRAM than RAM+swa

In [6]:
# actually do merge
import torch
import yaml

from mergekit.config import MergeConfiguration
from mergekit.merge import MergeOptions, run_merge

with open(CONFIG_YML, "r", encoding="utf-8") as fp:
    merge_config = MergeConfiguration.model_validate(yaml.safe_load(fp))

run_merge(
    merge_config,
    out_path=OUTPUT_PATH,
    options=MergeOptions(
        lora_merge_cache=LORA_MERGE_CACHE,
        cuda=torch.cuda.is_available(),
        copy_tokenizer=COPY_TOKENIZER,
        lazy_unpickle=LAZY_UNPICKLE,
        low_cpu_memory=LOW_CPU_MEMORY,
    ),
)
print("Done!")

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

Warmup loader cache:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Executing graph: 100%|██████████| 2048/2048 [00:34<00:00, 59.76it/s] 


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Done!


In [7]:
from huggingface_hub import HfApi, HfFolder, upload_folder

REPO_NAME = "TheMockingJay1013/gemma-3-sft-peft-dare-resta"  # Change to your Hugging Face repo
OUTPUT_PATH = "./merged"  # Folder where merged model is saved

# Create repo if it doesn't exist
api = HfApi()
api.create_repo(REPO_NAME, exist_ok=True)

# Upload merged model
upload_folder(
    folder_path=OUTPUT_PATH,
    repo_id=REPO_NAME,
    commit_message="Pushing merged model to Hugging Face Hub"
)

print(f"Model successfully uploaded to: https://huggingface.co/{REPO_NAME}")


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.60G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

Model successfully uploaded to: https://huggingface.co/TheMockingJay1013/gemma-3-sft-peft-dare-resta
